In [ ]:
%%html
<style>
#main{display:flex;gap:20px;align-items:flex-start;}
#left{display:flex;flex-direction:column;}
#controls{display:flex;gap:10px;margin-bottom:10px;align-items:center;flex-wrap:wrap;}
#right{width:300px;max-height:480px;overflow-y:auto;border-left:2px solid #ccc;padding-left:10px;}
video{width:640px;border:1px solid #ccc;}
canvas{position:absolute;left:0;top:0;}
#container{position:relative;width:640px;}
#snapshots{display:grid;grid-template-columns:repeat(2,1fr);gap:10px;}
.snap{border:1px solid #ccc;padding:4px;border-radius:6px;font-size:12px;text-align:center;}
.snap img{width:100%;border-radius:4px;}
.snap-time{margin:2px 0;}
.dl-btn{display:inline-block;margin-top:4px;padding:3px 8px;background:#333;color:#fff;
  border:none;border-radius:4px;cursor:pointer;font-size:11px;text-decoration:none;}
.dl-btn:hover{background:#555;}
.setting{display:flex;flex-direction:column;font-size:12px;gap:3px;}
.setting label{font-weight:bold;color:#444;}
.setting input{width:90px;padding:3px 6px;border:1px solid #aaa;border-radius:4px;font-size:13px;}
</style>

<div id="main">
  <div id="left">
    <div id="controls">
      <button id="startBtn">▶ Start</button>
      <button id="stopBtn">■ Stop</button>

      <div class="setting">
        <label>Suspicious Duration (ms)</label>
        <input type="number" id="durationInput" value="1000" min="200" max="10000" step="100">
      </div>

      <div class="setting">
        <label>Snapshot Cooldown (ms)</label>
        <input type="number" id="cooldownInput" value="1000" min="200" max="10000" step="100">
      </div>

      <div id="statusBadge" style="
        padding:4px 10px;border-radius:12px;font-size:12px;font-weight:bold;
        background:#eee;color:#555;align-self:flex-end;
      ">Idle</div>
    </div>

    <div id="container">
      <video id="video" autoplay playsinline></video>
      <canvas id="overlay"></canvas>
    </div>
  </div>
  <div id="right">
    <h3 style="margin-top:0;">Snapshots <span id="snapCount" style="font-size:13px;color:#888;">(0)</span></h3>
    <div id="snapshots"></div>
  </div>
</div>

<script type="module">
import {FilesetResolver, PoseLandmarker, FaceLandmarker, DrawingUtils}
  from "https://cdn.jsdelivr.net/npm/@mediapipe/tasks-vision@0.10.14";

const video       = document.getElementById("video");
const canvas      = document.getElementById("overlay");
const ctx         = canvas.getContext("2d");
const snapshotBox = document.getElementById("snapshots");
const statusBadge = document.getElementById("statusBadge");
const snapCount   = document.getElementById("snapCount");
const durationInput = document.getElementById("durationInput");
const cooldownInput = document.getElementById("cooldownInput");

let poseLandmarker, faceLandmarker, drawingUtils;
let running = false, rafId;
let audioCtx;
let suspiciousStartTime = null;
let lastSnapshotTime    = 0;
let totalSnaps          = 0;

function getSuspiciousDuration() {
  return Math.max(200, parseInt(durationInput.value) || 1000);
}
function getCooldown() {
  return Math.max(200, parseInt(cooldownInput.value) || 1000);
}

function setStatus(text, color, bg) {
  statusBadge.textContent = text;
  statusBadge.style.color = color;
  statusBadge.style.background = bg;
}

function beep() {
  if (!audioCtx) audioCtx = new (window.AudioContext || window.webkitAudioContext)();
  const osc = audioCtx.createOscillator();
  const gain = audioCtx.createGain();
  osc.frequency.value = 880;
  osc.connect(gain); gain.connect(audioCtx.destination);
  gain.gain.setValueAtTime(0.001, audioCtx.currentTime);
  gain.gain.exponentialRampToValueAtTime(0.2,  audioCtx.currentTime + 0.02);
  gain.gain.exponentialRampToValueAtTime(0.001, audioCtx.currentTime + 0.25);
  osc.start(); osc.stop(audioCtx.currentTime + 0.26);
}

function angle(a, b, c) {
  const abx = a.x-b.x, aby = a.y-b.y;
  const cbx = c.x-b.x, cby = c.y-b.y;
  const dot = abx*cbx + aby*cby;
  const mag1 = Math.hypot(abx, aby), mag2 = Math.hypot(cbx, cby);
  if (mag1 === 0 || mag2 === 0) return 0;
  return Math.acos(Math.max(-1, Math.min(1, dot/(mag1*mag2)))) * 180 / Math.PI;
}

function postureLabel(lm) {
  const lK = angle(lm[23], lm[25], lm[27]);
  const rK = angle(lm[24], lm[26], lm[28]);
  if (lK > 160 && rK > 160) return "Standing";
  if (lK < 140 && rK < 140) return "Sitting";
  return "Unknown";
}

function headDirection(lm) {
  const nose = lm[0], ls = lm[11], rs = lm[12];
  const mid = (ls.x + rs.x) / 2;
  const dx = nose.x - mid;
  if (dx >  0.06) return "Right";
  if (dx < -0.06) return "Left";
  return "Forward";
}

function eyeDirection(faceLm) {
  if (!faceLm[468] || !faceLm[473]) return "Eyes Forward";
  const dxL = faceLm[468].x - faceLm[33].x;
  const dxR = faceLm[473].x - faceLm[263].x;
  const avg = (dxL + dxR) / 2;
  if (avg >  0.015) return "Eyes Right";
  if (avg < -0.015) return "Eyes Left";
  return "Eyes Forward";
}

function getFaceBoundingBox(faceLm) {
  let minX=Infinity, minY=Infinity, maxX=-Infinity, maxY=-Infinity;
  for (const pt of faceLm) {
    if (pt.x < minX) minX = pt.x;
    if (pt.y < minY) minY = pt.y;
    if (pt.x > maxX) maxX = pt.x;
    if (pt.y > maxY) maxY = pt.y;
  }
  const pad = 0.015;
  return {
    x: (minX - pad) * canvas.width,
    y: (minY - pad) * canvas.height,
    w: (maxX - minX + pad*2) * canvas.width,
    h: (maxY - minY + pad*2) * canvas.height
  };
}

function captureSnapshot() {
  const off = document.createElement("canvas");
  off.width = canvas.width; off.height = canvas.height;
  const offCtx = off.getContext("2d");
  offCtx.drawImage(video, 0, 0);
  offCtx.drawImage(canvas, 0, 0);
  const img = off.toDataURL("image/png");
  const timeStr = new Date().toLocaleTimeString();

  totalSnaps++;
  snapCount.textContent = `(${totalSnaps})`;

  const card  = document.createElement("div");  card.className = "snap";
  const image = document.createElement("img");  image.src = img;
  const time  = document.createElement("div");
  time.className = "snap-time"; time.innerText = `#${totalSnaps} — ${timeStr}`;

  const dlBtn = document.createElement("a");
  dlBtn.className   = "dl-btn";
  dlBtn.href        = img;
  dlBtn.download    = "snapshot_" + timeStr.replace(/:/g, "-") + ".png";
  dlBtn.textContent = "⬇ Download";

  card.appendChild(image);
  card.appendChild(time);
  card.appendChild(dlBtn);
  snapshotBox.prepend(card);
}

async function initModels() {
  setStatus("Loading models…", "#333", "#fff3cd");
  const vision = await FilesetResolver.forVisionTasks(
    "https://cdn.jsdelivr.net/npm/@mediapipe/tasks-vision@0.10.14/wasm");
  poseLandmarker = await PoseLandmarker.createFromOptions(vision, {
    baseOptions: { modelAssetPath:
      "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task" },
    runningMode: "VIDEO", numPoses: 5 });
  faceLandmarker = await FaceLandmarker.createFromOptions(vision, {
    baseOptions: { modelAssetPath:
      "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task" },
    runningMode: "VIDEO", numFaces: 5 });
  drawingUtils = new DrawingUtils(ctx);
}

async function startCamera() {
  const stream = await navigator.mediaDevices.getUserMedia({ video: true });
  video.srcObject = stream;
  await new Promise(res => video.onloadedmetadata = res);
  canvas.width  = video.videoWidth;
  canvas.height = video.videoHeight;
}

function drawLabel(text, x, y) {
  ctx.fillStyle = "rgba(0,0,0,0.7)";
  ctx.fillRect(x, y, 300, 30);
  ctx.fillStyle = "white";
  ctx.font = "16px monospace";
  ctx.fillText(text, x+5, y+20);
}

async function loop() {
  if (!running) return;
  const now     = performance.now();
  const wallNow = Date.now();

  const poseResult = poseLandmarker.detectForVideo(video, now);
  const faceResult = faceLandmarker.detectForVideo(video, now);
  ctx.clearRect(0, 0, canvas.width, canvas.height);

  const poses = poseResult.landmarks || [];
  const faces = faceResult.faceLandmarks || [];
  let isSuspicious = false;

  poses.forEach((lm, i) => {
    drawingUtils.drawConnectors(lm, PoseLandmarker.POSE_CONNECTIONS);
    drawingUtils.drawLandmarks(lm);
    const posture = postureLabel(lm);
    const head    = headDirection(lm);
    let eye = "";
    if (faces.length > 0) eye = eyeDirection(faces[0]);
    drawLabel(`${posture} | Head:${head} | ${eye}`, 10, 40 + i*40);
    if (head === "Left" || head === "Right" ||
        eye  === "Eyes Left" || eye === "Eyes Right") isSuspicious = true;
  });

  faces.forEach(faceLm => {
    const bb = getFaceBoundingBox(faceLm);
    const head = poses.length > 0 ? headDirection(poses[0]) : "Forward";
    const eye  = eyeDirection(faceLm);
    const cheating = head === "Left" || head === "Right" ||
                     eye  === "Eyes Left" || eye === "Eyes Right";

    ctx.strokeStyle = cheating ? "#ff2222" : "#22ee44";
    ctx.lineWidth   = 3;
    ctx.shadowColor = cheating ? "rgba(255,30,30,0.6)" : "rgba(30,240,80,0.5)";
    ctx.shadowBlur  = 8;
    ctx.beginPath();
    ctx.roundRect(bb.x, bb.y, bb.w, bb.h, 6);
    ctx.stroke();
    ctx.shadowBlur = 0;

    const label = cheating ? "⚠ Cheating" : "✓ OK";
    ctx.font = "bold 14px monospace";
    const tw = ctx.measureText(label).width;
    ctx.fillStyle = cheating ? "rgba(200,0,0,0.75)" : "rgba(0,160,40,0.75)";
    ctx.beginPath();
    ctx.roundRect(bb.x, bb.y - 24, tw + 10, 20, 4);
    ctx.fill();
    ctx.fillStyle = "#ffffff";
    ctx.fillText(label, bb.x + 5, bb.y - 9);
  });

  if (isSuspicious) {
    setStatus("⚠ Suspicious", "#fff", "#dc3545");
    if (suspiciousStartTime === null) suspiciousStartTime = wallNow;
    const elapsed    = wallNow - suspiciousStartTime;
    const cooldownOk = (wallNow - lastSnapshotTime) >= getCooldown();
    if (elapsed >= getSuspiciousDuration() && cooldownOk) {
      beep();
      captureSnapshot();
      lastSnapshotTime    = wallNow;
      suspiciousStartTime = wallNow;
    }
  } else {
    setStatus("✓ Monitoring", "#fff", "#28a745");
    suspiciousStartTime = null;
  }

 rafId = requestAnimationFrame(loop);
}

document.getElementById("startBtn").onclick = async () => {
  if (running) return;
  // Lock inputs while running
  durationInput.disabled = true;
  cooldownInput.disabled = true;
  await initModels();
  await startCamera();
  running = true;
  setStatus("✓ Monitoring", "#fff", "#28a745");
  loop();
};

document.getElementById("stopBtn").onclick = () => {
  running = false;
  durationInput.disabled = false;
  cooldownInput.disabled = false;
  if (rafId) cancelAnimationFrame(rafId);
  if (video.srcObject) {
    video.srcObject.getTracks().forEach(t => t.stop());
    video.srcObject = null;
  }
  setStatus("Stopped", "#555", "#eee");
};
</script>

: 